# 트랜스포머 노드를 GPT로 변경하여 구현하기

이번 퀘스트는 지금까지 만들었던 Transformer(Encoder-Decoder) 코드를, GPT-1 논문("Improving Language Understanding by Generative Pre-Training", Radford et al., 2018)에 기반해서 Decoder-only 생성모델로 바꿔보는 퀘스트다. 챗봇 프로젝트에서 썼던 데이터를 재활용하되, 이번엔 지도학습(질문→답변)이 아니라 pretrain(다음 토큰 예측)만 다룬다.


## 1. Transformer 대비 변경이 필요한 부분 리스트업

번역기/챗봇 때 만들었던 Encoder-Decoder Transformer를 기준으로, GPT-1이 되려면 바꿔야 하는 부분들을 확인해본다. (실제 코드에서 바뀐 지점마다 `# [변경]` 주석을 달아뒀다.)

- **Encoder 제거 / Cross-Attention 제거**: GPT는 다른 시퀀스를 참조할 일이 없고 지금까지 나온 텍스트만 보고 다음 토큰을 예측한다. 그래서 인코더가 통째로 빠지고, 인코더 출력을 참조하던 Cross-Attention도 같이 사라진다. 결국 디코더의 Masked Self-Attention 블록만 남게되는 형식이다.

- **마스크가 하나로 통일됨**: 원래는 인코더용(패딩만) / 디코더용(패딩+causal) 마스크가 따로였는데, 블록이 디코더 하나만 남으니 (패딩 + causal) 마스크 하나로 충분하다.

- **위치 정보를 학습형으로**: 번역기에서는 sin/cos로 만든 고정 positional encoding을 썼는데, GPT는 위치 임베딩도 학습되는 파라미터(`nn.Embedding`)로 두어야 한다.

- **weight tying**: 출력층 가중치를 입력 토큰 임베딩과 공유한다. 논문 수식 `P(u) = softmax(hn·We^T)`에서 출력 projection이 임베딩 행렬 `We`를 그대로 재사용하는 형태다.

- **학습 목표가 바뀜**: 소스→타겟 번역(seq2seq)이 아니라, 텍스트 하나를 주고 다음 단어를 맞히는 언어모델링이다.

- **기타**: 활성함수는 ReLU 대신 GELU(논문 4.1절), 가중치 초기화는 N(0, 0.02)를 쓴다. (초기화는 안 지켰다가 실제로 문제가 생겼었다.)


## 2. 라이브러리 및 데이터 준비

In [1]:
!pip install -q python-mecab-ko

import re, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from mecab import MeCab

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


챗봇 프로젝트 때 썼던 것과 같은 데이터([songys/Chatbot_data](https://github.com/songys/Chatbot_data))를 재사용한다.

In [2]:
DATA_URL = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
data = pd.read_csv(DATA_URL)

questions = data["Q"].tolist()
answers = data["A"].tolist()
print("데이터 크기:", data.shape)

데이터 크기: (11823, 3)


## 3. 모델의 입력 형태에 맞게 전처리 수행

지시사항 2번("Decoder 기반 생성모델임을 감안하여 챗봇 데이터를 변형")에 해당하는 부분이다.

지도학습(챗봇 프로젝트)에서는 질문과 답변을 분리된 두 시퀀스(`enc_train`/`dec_train`)로 다뤘지만, GPT는 인코더-디코더 구분이 없는 단일 시퀀스 모델이라 나눠서 넣을 수 없다. 그래서 질문과 답변을 `<sep>`으로 이어붙여 하나의 연속 시퀀스로 만든다. "질문 다음에 sep, sep 다음에 답변"이라는 패턴 자체를 다음 토큰 예측으로 학습하게 되는, pretrain 방식 그대로다.

In [3]:
def preprocess_sentence(sentence):
    sentence = str(sentence).lower().strip()
    sentence = re.sub(r"[^a-z0-9ㄱ-ㅎㅏ-ㅣ가-힣?.!,]+", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence

mecab = MeCab()

# 질문 + <sep> + 답변을 하나의 시퀀스로 이어붙인다 (GPT는 소스/타겟 구분이 없다)
corpus = []
for q, a in zip(questions, answers):
    q_tok = mecab.morphs(preprocess_sentence(q))
    a_tok = mecab.morphs(preprocess_sentence(a))
    corpus.append(q_tok + ["<sep>"] + a_tok)

print("코퍼스 예시:", corpus[0])
print("총 시퀀스 수:", len(corpus))

코퍼스 예시: ['12', '시', '땡', '!', '<sep>', '하루', '가', '또', '가', '네요', '.']
총 시퀀스 수: 11823


단어 사전을 만들고, 각 시퀀스 앞뒤에 `<bos>`/`<eos>`를 붙여서 정수 인덱스로 벡터화한다. (`<sep>`도 이제 일반 어휘처럼 취급 — 소스/타겟을 나누는 특수 토큰이 아니라, 문장 안에 등장하는 하나의 "구분자 단어"로 다룬다는 점이 챗봇 프로젝트와 다른 지점이다.)

In [4]:
from collections import Counter

PAD, UNK, BOS, EOS, SEP = "<pad>", "<unk>", "<bos>", "<eos>", "<sep>"

counter = Counter(tok for sent in corpus for tok in sent if tok != SEP)
vocab = [PAD, UNK, BOS, EOS, SEP] + [w for w, _ in counter.most_common()]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(vocab)
print("VOCAB_SIZE:", VOCAB_SIZE)

MAX_LEN = 40

def vectorize(corpus, maxlen):
    arr = np.zeros((len(corpus), maxlen), dtype=np.int64)
    for i, sent in enumerate(corpus):
        ids = [word2idx[BOS]] + [word2idx.get(t, word2idx[UNK]) for t in sent] + [word2idx[EOS]]
        ids = ids[:maxlen]
        arr[i, :len(ids)] = ids
    return arr

data_ids = vectorize(corpus, MAX_LEN)
print("data_ids shape:", data_ids.shape)
print("예시:", data_ids[0][:15])

VOCAB_SIZE: 6842
data_ids shape: (11823, 40)
예시: [   2 2341  175 3341   89    4  248   11  147   11   37    5    3    0
    0]


## 4. 모델의 입력 블럭을 GPT 논문에 기반하여 수정

논문 3.1절의 입력 수식은 `h0 = U·We + Wp` 이다.

- `U`는 입력 토큰 인덱스, `We`는 토큰 임베딩 행렬, `Wp`는 위치 임베딩 행렬이다.
- 여기서 `Wp`가 번역기에서 쓰던 sin/cos 고정 위치 인코딩과 다른 부분인데, GPT는 위치 정보도 학습되는 파라미터로 둔다.

`nn.Embedding`을 토큰용/위치용 두 개 만들어서 더해준다. 이때 위치 임베딩의 사전 크기는 시퀀스 최대 길이(`MAX_LEN`)로 잡는다. 단어가 아니라 "0번째 위치, 1번째 위치..."를 각각 하나의 인덱스로 취급하는 셈이다.

In [5]:
# 논문 3.1절 h0 = U*We + Wp : 토큰 임베딩 + 위치 임베딩
class TokenPositionEmbedding(nn.Module):
    # [변경] Transformer는 sin/cos 고정 위치 인코딩 -> GPT-1은 위치도 학습 파라미터로 둠
    def __init__(self, vocab_size, d_model, max_len, dropout):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)   # [변경] 학습되는 위치 임베딩 (Wp)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)  # (1, L)
        out = self.tok_emb(x) + self.pos_emb(positions)                  # 브로드캐스팅으로 (B, L, d_model)
        return self.dropout(out)


# 입력이 정상적으로 구성되는지 확인 (지시사항: "모델의 input이 정상적으로 구성되었는지 확인")
_test_emb = TokenPositionEmbedding(VOCAB_SIZE, d_model=16, max_len=MAX_LEN, dropout=0.0)
_test_x = torch.tensor(data_ids[:2])
_test_out = _test_emb(_test_x)
print("입력 시퀀스 shape:", _test_x.shape)
print("임베딩 출력 shape:", _test_out.shape, "(batch, seq_len, d_model)")

입력 시퀀스 shape: torch.Size([2, 40])
임베딩 출력 shape: torch.Size([2, 40, 16]) (batch, seq_len, d_model)


## 5. GPT-1 모델 구성

Transformer의 `DecoderLayer`에서 Cross-Attention을 제거하면 `GPTBlock`이 된다. Masked Self-Attention(causal + padding mask) → FFN, 이렇게 두 개만 남는다.

활성함수는 논문에 명시된 대로 ReLU 대신 **GELU**를 쓰고, 출력층은 토큰 임베딩과 **weight tying**한다 (`P(u) = softmax(hn*We^T)`).

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, L, _ = x.size()
        return x.view(B, L, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, x, mask):
        q = self.split_heads(self.wq(x))
        k = self.split_heads(self.wk(x))
        v = self.split_heads(self.wv(x))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        scores = scores.masked_fill(mask, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)

        B, H, L, Dh = out.size()
        out = out.transpose(1, 2).contiguous().view(B, L, H * Dh)
        return self.wo(out)


# [변경] Transformer DecoderLayer는 (Self-Attn -> Cross-Attn -> FFN) 3단계였는데,
# GPT는 참조할 인코더 출력이 없어서 Cross-Attention을 빼고 2단계만 남김
class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),                 # [변경] Transformer: ReLU -> GPT-1: GELU (논문 4.1절)
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # [변경] Masked Self-Attention 하나만 (Cross-Attention 제거됨)
        x = self.norm1(x + self.dropout(self.attn(x, mask)))
        # FFN
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class GPT1(nn.Module):
    def __init__(self, vocab_size, n_layers, d_model, n_heads, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.embed = TokenPositionEmbedding(vocab_size, d_model, max_len, dropout)
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size, bias=False)
        # [변경] weight tying: 출력층 가중치를 입력 토큰 임베딩과 공유 (논문 P(u)=softmax(hn*We^T))
        self.fc_out.weight = self.embed.tok_emb.weight

    def make_causal_pad_mask(self, x):
        # 패딩 마스크 + causal(미래 차단) 마스크를 OR로 결합
        # [변경] Transformer는 인코더/디코더 마스크가 따로였는데, GPT는 이거 하나로 통일됨
        B, L = x.size()
        pad = (x == 0).unsqueeze(1).unsqueeze(2)                                        # (B,1,1,L)
        future = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), diagonal=1)  # (L,L)
        return pad | future

    def forward(self, x):
        mask = self.make_causal_pad_mask(x)
        h = self.embed(x)
        for block in self.blocks:
            h = block(h, mask)
        return self.fc_out(h)

### 하이퍼파라미터 및 가중치 초기화

논문 원본 스펙(12층, d_model=768, head 12개)은 우리 데이터/자원엔 과해서 층 수랑 차원을 줄였다.

가중치 초기화는 논문 4.1절대로 N(0, 0.02)로 맞췄는데, 이걸 처음에 안 하고 PyTorch 기본값으로 돌렸다가 한 번 헤맸다. weight tying으로 임베딩 행렬을 출력층에도 같이 쓰다 보니 logit이 커져서, loss가 정상 범위(`ln(vocab_size)` 근처)보다 한참 높은 값에서 시작했다. N(0, 0.02)로 좁혀주니 바로 정상으로 돌아왔다.

In [7]:
N_LAYERS = 4
D_MODEL = 256
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.1

model = GPT1(VOCAB_SIZE, N_LAYERS, D_MODEL, N_HEADS, D_FF, MAX_LEN, dropout=DROPOUT).to(device)

# [변경] Transformer: PyTorch 기본 초기화 -> GPT-1: N(0, 0.02) (논문 4.1절)
def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

model.apply(init_weights)
model.fc_out.weight = model.embed.tok_emb.weight  # 초기화 후 weight tying 다시 연결 (apply가 개별적으로 덮어씀)

print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("파라미터 수:", n_params)

GPT1(
  (embed): TokenPositionEmbedding(
    (tok_emb): Embedding(6842, 256, padding_idx=0)
    (pos_emb): Embedding(40, 256)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (blocks): ModuleList(
    (0-3): 4 x GPTBlock(
      (attn): MultiHeadAttention(
        (wq): Linear(in_features=256, out_features=256, bias=True)
        (wk): Linear(in_features=256, out_features=256, bias=True)
        (wv): Linear(in_features=256, out_features=256, bias=True)
        (wo): Linear(in_features=256, out_features=256, bias=True)
      )
      (ffn): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=1024, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (fc_out): Linear(in_f

## 6. 훈련하기

지도학습이 아니라 순수 언어모델링이라 훨씬 단순했다. 시퀀스를 한 칸씩 어긋나게 잘라서(`x[:-1]`이 입력, `x[1:]`이 정답) 다음 토큰을 맞히도록 학습한다.

In [8]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
EPOCHS = 10

ids_tensor = torch.from_numpy(data_ids).to(device)
loader = DataLoader(TensorDataset(ids_tensor), batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=2.5e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # <pad> 위치는 손실 계산에서 제외

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    n_batch = 0
    for (batch,) in loader:
        x_in = batch[:, :-1]    # <bos> w1 w2 ... (입력)
        x_out = batch[:, 1:]    # w1 w2 ... <eos> (정답, 한 칸 밀림)

        optimizer.zero_grad()
        logits = model(x_in)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), x_out.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batch += 1

    print(f"Epoch {epoch:2d} | loss {total_loss / n_batch:.4f}")

Epoch  1 | loss 5.4895
Epoch  2 | loss 4.1422
Epoch  3 | loss 3.7859
Epoch  4 | loss 3.5823
Epoch  5 | loss 3.4247
Epoch  6 | loss 3.2910
Epoch  7 | loss 3.1738
Epoch  8 | loss 3.0676
Epoch  9 | loss 2.9688
Epoch 10 | loss 2.8733


## 7. 입력에 따른 출력 생성

Greedy decoding으로 프롬프트 뒤에 이어질 텍스트를 생성한다. 평가기준 5번이 "출력 수준에 상관없이 정상 동작 확인"이므로, 파이프라인이 에러 없이 끝까지 도는지가 핵심이다.

In [9]:
@torch.no_grad()
def generate(prompt, model, max_new_tokens=20):
    model.eval()
    tokens = mecab.morphs(preprocess_sentence(prompt))
    ids = [word2idx[BOS]] + [word2idx.get(t, word2idx[UNK]) for t in tokens]
    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        logits = model(x[:, -MAX_LEN:])          # MAX_LEN을 넘지 않도록 최근 토큰만 사용
        next_id = int(logits[0, -1].argmax())
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)
        if next_id == word2idx[EOS]:
            break

    out_ids = x[0].tolist()
    words = [idx2word[i] for i in out_ids if i not in (word2idx[PAD], word2idx[BOS])]
    model.train()
    return " ".join(words)


examples = [
    "오늘 기분이 좋아",
    "지루하다, 놀러가고 싶어",
    "간만에 여자친구랑 데이트 하기로 했어",
    "집에 있는다는 소리야",
]

for ex in examples:
    print(f"입력: {ex}")
    print(f"생성: {generate(ex, model)}")
    print()

입력: 오늘 기분이 좋아
생성: 오늘 기분 이 좋 아 <sep> 저 도 좋 아 해요 . <eos>

입력: 지루하다, 놀러가고 싶어
생성: 지루 하 다 , 놀 러 가 고 싶 어 <sep> 같이 가 보 세요 . <eos>

입력: 간만에 여자친구랑 데이트 하기로 했어
생성: 간만에 여자 친구 랑 데이트 하 기 로 했 어 <sep> 운동 을 해 보 세요 . <eos>

입력: 집에 있는다는 소리야
생성: 집 에 있 는다는 <unk> 좋 은 날 이 안 되 는 거 같 아 <sep> 좋 은 곳 으로 가 보 는 게 좋



## 회고

이전 퀘스트에서 Transformer를 직접 짜봐서 GPT-1은 좀 만만하게 봤는데, 막상 해보니 생각보다 쉽지 않았다. "뭘 빼야 하는지"를 정확히 아는 게 관건이었다.   
Encoder랑 Cross-Attention 들어내는 건 금방이었는데, 위치 인코딩을 학습형으로 바꾸고 출력층을 임베딩이랑 묶는 weight tying 같은 건 논문을 대충 읽으면 놓치기 쉬웠다.
특히 초기화 문제에서 시간이 좀 지체됐다. weight tying을 쓰면 임베딩 행렬이 입력에도 출력에도 같이 쓰이는데, 기본 초기화로는 값이 너무 커서 loss가 이상하게 높게 시작됐다. 논문이 실험 세팅이라고 적어둔 N(0, 0.02) 같은 값들이 그냥 넣은 게 아니라는 걸 직접 해보고 나서 알았다.  

결과는 생각보다 괜찮았다. pretrain만 10 epoch 돌렸는데 loss가 5.49에서 2.87까지 떨어졌고, <sep> 뒤에 입력과 얼추 관련된 답변 형태가 이어지면서 <eos>로 끝맺었다. 완벽한 대답까진 아니어도 "질문 다음엔 답이 온다"는 구조 자체는 학습된 것으로 보인다.